# Celabot — Paso 3: detección de *concealment* con pose

Construimos la detección estrella de hurto en tienda: el momento en que una persona toma un producto y lo guarda en su ropa o bolso en vez del carrito.

## El patrón que buscamos

Cuando alguien guarda algo en el carrito, su brazo vuelve a salir. Cuando lo guarda en el bolsillo o bolso, la mano **se queda cerca del cuerpo**. Esa permanencia es la señal.

```
distancia mano-cuerpo
       ▲
   alta│     ▄▀▀▄                      ← brazo extendido (toma producto)
       │    ▀    ▀▄
   baja│            ▄▄▄▄▄▄▄▄▄▄▄▄▄▄    ← mano regresa al cuerpo y se queda
       └──────────────────────────────► tiempo
            ↑       ↑
         reach    return + dwell
```

## Filosofía

Esta heurística por sí sola **tiene muchos falsos positivos** (guardar el celular, sacar la billetera, rascarse). No la usamos para alertar — la usamos como **filtro de candidatos** para que el VLM solo vea momentos sospechosos en vez de todo el video. El plan §3.3 al pie de la letra.

## Lo que aprendes

- **YOLO-pose** y los 17 *keypoints* COCO (nariz, hombros, muñecas, caderas...).
- **Normalización por torso**: distancias en "largos de torso" → independiente de la cámara.
- **State machine temporal** para detectar el patrón *reach → return → dwell*.
- Por qué pose es **necesaria pero no suficiente** y por qué siempre necesitas un VLM o modelo de acción encima.

## 0. Setup

Mismas dependencias que `zones.ipynb`. YOLO-pose viene en el mismo paquete `ultralytics`.

In [ ]:
!pip install -q ultralytics supervision google-genai opencv-python-headless matplotlib

## 1. Los 17 *keypoints* COCO

YOLO-pose devuelve por cada persona 17 puntos `(x, y, confidence)`. Los que nos importan:

| Índice | Nombre | Para qué lo usamos |
|---|---|---|
| 5 | hombro izq. | referencia de torso |
| 6 | hombro der. | referencia de torso |
| 9 | muñeca izq. | la "mano" |
| 10 | muñeca der. | la "mano" |
| 11 | cadera izq. | referencia de torso |
| 12 | cadera der. | referencia de torso |

**Centro del cuerpo** = punto medio entre el centro de hombros y el centro de caderas. 
**Largo de torso** = distancia entre esos dos centros. Usamos esto como unidad de medida para que el sistema funcione igual con la persona cerca o lejos de la cámara.

**Distancia mano-cuerpo normalizada** = `|muñeca - centro_cuerpo| / largo_torso`. Valores típicos: 0.3 = mano descansando junto al cuerpo, 0.8 = brazo extendido, >1.0 = brazo completamente extendido o levantado.

## 2. Sube tu video

Idealmente uno donde:
- Te grabas a ti mismo simulando el patrón: tomas un objeto de una mesa, lo metes al bolsillo, dejas la mano ahí 3-5 s.
- Para comparar, en otra parte del video sacas el objeto y lo dejas en la mesa (la mano vuelve a salir).
- Cámara fija, persona visible de cintura para arriba, resolución ≥ 480p.

In [ ]:
from google.colab import files
import cv2

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'Video: {VIDEO_PATH}  {W}x{H}  fps={FPS:.1f}  dur={TOTAL_FRAMES/FPS:.1f}s')

## 3. YOLO-pose en un solo frame

Antes de procesar todo el video, vemos qué retorna el modelo en un frame intermedio.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

pose_model = YOLO('yolo11n-pose.pt')

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, TOTAL_FRAMES // 2)
_, mid_frame = cap.read()
cap.release()

results = pose_model(mid_frame, verbose=False)[0]
annotated = results.plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('YOLO-pose: esqueleto con 17 keypoints')
plt.show()

n_people = len(results.boxes) if results.boxes is not None else 0
print(f'Personas detectadas: {n_people}')
if n_people > 0 and results.keypoints is not None:
    kp = results.keypoints.xy[0].cpu().numpy()  # primera persona
    kp_c = results.keypoints.conf[0].cpu().numpy()
    NAMES = ['nariz','ojo_i','ojo_d','oreja_i','oreja_d','hombro_i','hombro_d','codo_i','codo_d','muñeca_i','muñeca_d','cadera_i','cadera_d','rodilla_i','rodilla_d','tobillo_i','tobillo_d']
    for i in [5,6,9,10,11,12]:
        print(f'  {i:2d} {NAMES[i]:10s} ({kp[i,0]:6.0f}, {kp[i,1]:6.0f})  conf={kp_c[i]:.2f}')

## 4. La métrica clave: distancia mano-cuerpo normalizada

Función pura que toma keypoints y devuelve la distancia normalizada de cada muñeca al centro del cuerpo. Retorna `None` si los keypoints necesarios no están visibles (la persona está parcialmente oculta, de espaldas, etc.).

In [ ]:
import numpy as np

KP_LSHOULDER, KP_RSHOULDER = 5, 6
KP_LWRIST, KP_RWRIST = 9, 10
KP_LHIP, KP_RHIP = 11, 12
KP_CONF_MIN = 0.3

def wrist_body_distances(kp_xy, kp_conf):
    """Devuelve (d_izq, d_der, body_center, torso) normalizadas. None si no calculable."""
    needed = [KP_LSHOULDER, KP_RSHOULDER, KP_LHIP, KP_RHIP]
    if not all(kp_conf[k] >= KP_CONF_MIN for k in needed):
        return None, None, None, None
    mid_shoulder = (kp_xy[KP_LSHOULDER] + kp_xy[KP_RSHOULDER]) / 2
    mid_hip = (kp_xy[KP_LHIP] + kp_xy[KP_RHIP]) / 2
    body_center = (mid_shoulder + mid_hip) / 2
    torso = float(np.linalg.norm(mid_shoulder - mid_hip))
    if torso < 1:
        return None, None, None, None
    def d(idx):
        if kp_conf[idx] < KP_CONF_MIN:
            return None
        return float(np.linalg.norm(kp_xy[idx] - body_center) / torso)
    return d(KP_LWRIST), d(KP_RWRIST), body_center, torso

# Sanity check sobre el frame del medio
if n_people > 0 and results.keypoints is not None:
    l, r, _, t = wrist_body_distances(kp, kp_c)
    print(f'Torso = {t:.0f} px')
    print(f'Muñeca izq: {l}  (en largos de torso)')
    print(f'Muñeca der: {r}  (en largos de torso)')

## 5. Carril rápido: pose + tracking + acumulación de señal

Usamos `model.track(persist=True)` que combina detección, pose y tracking en una sola llamada y nos asigna un `id` estable a cada persona. Por cada frame, para cada persona, registramos `(timestamp, distancia_mano_cuerpo_min)` donde tomamos la muñeca más cercana al cuerpo (porque puede haber una mano visible y la otra no).

El callback dibuja además dos cosas extra encima del esqueleto:
- Un punto **verde** en cada muñeca si está extendida (`d > 0.6`).
- Un punto **rojo** en cada muñeca si está pegada al cuerpo (`d < 0.45`).

Procesar todos los frames sin GPU puede tomar 1–3 min por minuto de video. Con T4 baja a ~10–20 s/min.

In [ ]:
import supervision as sv

track_signals: dict[int, list[tuple[float, float]]] = {}  # tid -> [(ts, d_min), ...]

HIGH = 0.6
LOW = 0.45

def callback(frame, frame_idx):
    ts = frame_idx / FPS
    results = pose_model.track(frame, persist=True, classes=[0], tracker='bytetrack.yaml', verbose=False)[0]
    annotated = results.plot()

    if results.boxes is None or results.boxes.id is None or results.keypoints is None:
        return annotated

    ids = results.boxes.id.cpu().numpy().astype(int)
    kps_xy = results.keypoints.xy.cpu().numpy()
    kps_conf = (results.keypoints.conf.cpu().numpy() if results.keypoints.conf is not None
                else np.ones((kps_xy.shape[0], 17)))

    for i, tid in enumerate(ids):
        l, r, _, _ = wrist_body_distances(kps_xy[i], kps_conf[i])
        dists = [d for d in (l, r) if d is not None]
        if not dists:
            continue
        d_min = min(dists)
        track_signals.setdefault(int(tid), []).append((ts, d_min))

        # Overlay verde/rojo en las muñecas visibles
        for kp_idx, d in ((KP_LWRIST, l), (KP_RWRIST, r)):
            if d is None:
                continue
            x, y = int(kps_xy[i, kp_idx, 0]), int(kps_xy[i, kp_idx, 1])
            if d >= HIGH:
                cv2.circle(annotated, (x, y), 10, (0, 255, 0), -1)
            elif d <= LOW:
                cv2.circle(annotated, (x, y), 10, (0, 0, 255), -1)

    return annotated

print('Procesando video con pose + tracking (paciencia sin GPU)...')
sv.process_video(source_path=VIDEO_PATH, target_path='annotated_raw.mp4', callback=callback)
print(f'Listo. Tracks con señal: {len(track_signals)}')
for tid, sig in track_signals.items():
    print(f'  tid #{tid}: {len(sig)} muestras')

## 6. Mira el video anotado

Re-codificamos para reproducción en browser y lo embebemos. Fíjate en los círculos verde/rojo sobre las muñecas — vas a ver el patrón visualmente.

In [ ]:
import subprocess
from IPython.display import HTML
from base64 import b64encode

subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', 'annotated_raw.mp4',
    '-c:v', 'libx264', '-preset', 'ultrafast', '-movflags', '+faststart', '-an',
    'annotated.mp4',
], check=True)
mp4 = open('annotated.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=720 controls><source src="{data_url}" type="video/mp4"></video>')

## 7. Detectar el arco *reach → return → dwell*

State machine sobre la señal por track. Cuatro estados:

1. `searching` — esperando ver una mano extendida.
2. `after_reach` — vimos `d ≥ HIGH`; esperando que regrese cerca del cuerpo.
3. `in_low` — `d ≤ LOW`; midiendo cuánto se queda.
4. Si el dwell `≥ DWELL_S`, emitimos un candidato.

Si pasan más de 10 s en `after_reach` sin volver, reseteamos (probablemente no era una toma de objeto). Si pasan más de 8 s en `in_low` sin salir, también emitimos (la mano se quedó realmente quieta).

Suavizamos la señal con una media móvil corta para evitar disparos por ruido de detección.

In [ ]:
DWELL_S = 2.5
REACH_TIMEOUT_S = 10.0
DWELL_MAX_S = 8.0
SMOOTH_WINDOW = 5  # frames muestreados

def smooth(values, w):
    if len(values) < w:
        return values
    out = []
    half = w // 2
    for i in range(len(values)):
        lo, hi = max(0, i - half), min(len(values), i + half + 1)
        out.append(sum(values[lo:hi]) / (hi - lo))
    return out

def detect_arcs(signal):
    if not signal:
        return []
    ts = [s[0] for s in signal]
    d = smooth([s[1] for s in signal], SMOOTH_WINDOW)
    arcs = []
    state = 'searching'
    reach_ts = None
    low_ts = None
    for i, (t, v) in enumerate(zip(ts, d)):
        if state == 'searching':
            if v >= HIGH:
                state, reach_ts = 'after_reach', t
        elif state == 'after_reach':
            if v <= LOW:
                state, low_ts = 'in_low', t
            elif t - reach_ts > REACH_TIMEOUT_S:
                state, reach_ts = 'searching', None
        elif state == 'in_low':
            if v > LOW + 0.1:  # mano volvió a salir
                if t - low_ts >= DWELL_S:
                    arcs.append({'reach_ts': reach_ts, 'low_ts': low_ts, 'end_ts': t, 'dwell': t - low_ts})
                state, reach_ts, low_ts = 'searching', None, None
            elif t - low_ts >= DWELL_MAX_S:  # se quedó demasiado tiempo
                arcs.append({'reach_ts': reach_ts, 'low_ts': low_ts, 'end_ts': t, 'dwell': t - low_ts})
                state, reach_ts, low_ts = 'searching', None, None
    # cerrar si terminamos en in_low con dwell suficiente
    if state == 'in_low' and ts[-1] - low_ts >= DWELL_S:
        arcs.append({'reach_ts': reach_ts, 'low_ts': low_ts, 'end_ts': ts[-1], 'dwell': ts[-1] - low_ts})
    return arcs

concealment_candidates = []
for tid, signal in track_signals.items():
    for arc in detect_arcs(signal):
        concealment_candidates.append({
            'tid': tid,
            'start': max(0, arc['reach_ts'] - 1),
            'end': arc['end_ts'] + 1,
            **arc,
        })

print(f'Candidatos de concealment: {len(concealment_candidates)}')
for c in concealment_candidates:
    print(f"  tid #{c['tid']}: reach {c['reach_ts']:.1f}s → low {c['low_ts']:.1f}s → end {c['end_ts']:.1f}s (dwell {c['dwell']:.1f}s)")

## 8. Visualiza la señal del candidato

Si hay candidatos, graficamos la distancia mano-cuerpo del primero a lo largo del tiempo, con líneas para los umbrales y los eventos del arco detectado. Esto es lo que vas a ver en el dashboard interno cuando depures alertas.

In [ ]:
import matplotlib.pyplot as plt

if concealment_candidates:
    c = concealment_candidates[0]
    signal = track_signals[c['tid']]
    ts_arr = [s[0] for s in signal]
    raw = [s[1] for s in signal]
    sm = smooth(raw, SMOOTH_WINDOW)

    plt.figure(figsize=(12, 4))
    plt.plot(ts_arr, raw, alpha=0.3, label='raw')
    plt.plot(ts_arr, sm, label='suavizada')
    plt.axhline(HIGH, color='green', linestyle='--', label=f'HIGH={HIGH}')
    plt.axhline(LOW, color='red', linestyle='--', label=f'LOW={LOW}')
    plt.axvspan(c['low_ts'], c['end_ts'], color='red', alpha=0.15, label='dwell')
    plt.axvline(c['reach_ts'], color='green', alpha=0.7)
    plt.xlabel('tiempo (s)')
    plt.ylabel('|muñeca - centro| / torso')
    plt.title(f"Track #{c['tid']} — arco de concealment")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    # Mostrar la señal del primer track aunque no haya disparo, para entender qué pasó
    if track_signals:
        tid = next(iter(track_signals))
        signal = track_signals[tid]
        ts_arr = [s[0] for s in signal]
        sm = smooth([s[1] for s in signal], SMOOTH_WINDOW)
        plt.figure(figsize=(12, 4))
        plt.plot(ts_arr, sm)
        plt.axhline(HIGH, color='green', linestyle='--')
        plt.axhline(LOW, color='red', linestyle='--')
        plt.title(f'Track #{tid} — sin arco detectado')
        plt.xlabel('tiempo (s)'); plt.ylabel('distancia normalizada')
        plt.grid(alpha=0.3)
        plt.show()
    else:
        print('No hubo pose detectada con suficiente confianza en ningún frame.')

## 9. Carril lento: Gemini confirma o descarta

Helpers idénticos a `zones.ipynb`, con un *prompt* específico para concealment que enumera explícitamente comportamientos normales (anti-falsos-positivos) y la categoría que esperamos.

In [ ]:
import os, subprocess, time, json, getpass
from google import genai

def extract_clip(src, start, end, out):
    if os.path.exists(out):
        os.remove(out)
    duration = max(0.5, end - start)
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', src, '-ss', f'{start:.2f}', '-t', f'{duration:.2f}',
        '-c:v', 'libx264', '-preset', 'ultrafast', '-movflags', '+faststart', '-an',
        out,
    ], check=True)
    return out

def ask_gemini(client, clip_path, prompt, model='gemini-2.5-flash'):
    vf = client.files.upload(file=clip_path)
    while vf.state.name == 'PROCESSING':
        time.sleep(2)
        vf = client.files.get(name=vf.name)
    resp = client.models.generate_content(model=model, contents=[vf, prompt])
    raw = resp.text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

GEMINI_API_KEY = getpass.getpass('Pega tu API key de Gemini: ')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
PROMPT_CONCEALMENT = '''Eres un analista de seguridad de tienda. Una heurística de pose detectó que una persona extendió el brazo (posiblemente tomó algo) y luego mantuvo la mano cerca del cuerpo varios segundos (posible ocultamiento).

Mira el clip y responde ESTRICTAMENTE en JSON:
{"anomaly": bool, "confidence": float, "category": "ocultamiento" | "normal", "observed_behaviors": [str], "reason": str, "suggested_action": str}

Reglas:
- NORMAL: guardar/sacar celular, billetera o llaves; rascarse; ajustarse ropa; meter mano al carrito; sostener un producto contra el cuerpo para verlo.
- SOSPECHOSO: meter un producto al bolsillo del pantalón/abrigo/bolso, esconder bajo la ropa, transferir un ítem disimulando el movimiento.
- Si no es claro, anomaly=false con confidence baja.
- Solo el JSON, sin texto extra.'''

if not concealment_candidates:
    print('Sin candidatos. Prueba con un video donde explícitamente metas algo al bolsillo, o baja DWELL_S y HIGH.')
else:
    for i, c in enumerate(concealment_candidates[:3]):  # primeros 3 para no quemar cuota
        print(f"\n=== Candidato {i+1} (tid #{c['tid']}) ===")
        end = min(c['end'], c['start'] + 12)
        clip = f'clip_concealment_{i}.mp4'
        extract_clip(VIDEO_PATH, c['start'], end, clip)
        try:
            verdict = ask_gemini(client, clip, PROMPT_CONCEALMENT)
            print(json.dumps(verdict, indent=2, ensure_ascii=False))
        except Exception as e:
            print(f'Error: {e}')

## 10. Limitaciones honestas de este enfoque

Esto **no es un detector de robo terminado**. Funciona como filtro de candidatos. Sus debilidades reales:

1. **Ángulo de cámara**: si la persona está de espaldas, no vemos sus muñecas hacia adelante; si está perfilada, la métrica `|muñeca - centro| / torso` se distorsiona. Mitigación: cámaras a 45° respecto a las góndolas.
2. **Oclusiones**: carritos, mochilas, otra persona delante → keypoints faltantes → no detectamos el arco. Mitigación: múltiples cámaras + sensor fusion.
3. **Falsos positivos legítimos**: guardar celular, sacar billetera, abrigos pesados donde la mano siempre está cerca del cuerpo. Mitigación: el VLM filtra, y el botón "falsa alarma" del dueño (§17 Tier 1) alimenta el siguiente entrenamiento.
4. **No ve lo que hay en la mano**: no distinguimos "metió producto al bolsillo" de "metió las manos al bolsillo". Eso lo resuelve el VLM al ver el clip.
5. **Sensible a umbrales**: `HIGH`, `LOW`, `DWELL_S` deben afinarse por tipo de tienda (un OXXO no es una boutique). Mitigación: aprender los umbrales por tienda a partir de las primeras semanas de etiquetas Tier 1.

## Qué cambió respecto al paso 2

| | Paso 2 (zonas) | Paso 3 (concealment) |
|---|---|---|
| Modelo | YOLO11 detección | **YOLO11-pose** (17 keypoints) |
| Señal espacial | dentro/fuera de zona, cruce de línea | **trayectoria de la muñeca** respecto al cuerpo |
| Estado por track | tiempo en zona | **state machine** con 3 estados |
| Visualización | bbox + ID + trayectoria | + esqueleto + muñecas verdes/rojas |
| Casos de uso del plan | intrusión, merodeo, grab-and-run | **concealment** (#3 del plan §2) |

## Siguiente paso

Dos caminos para el **paso 4**:

1. **RTSP en vivo** — cambiar `VIDEO_PATH` por un stream `rtsp://...`, simular ingesta real con `mediaMTX`, medir latencia end-to-end. Es el siguiente salto arquitectural del plan.
2. **Detector de arma** — fine-tune YOLO11 con un dataset público; conectar con un prompt de Gemini específico para casos críticos (latencia objetivo < 5 s). Es el caso de uso #1 del plan §2.

Dime cuál sigue.